In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig
from esm.utils.constants.esm3 import (
    SEQUENCE_MASK_TOKEN,
)
import torch
device = torch.device('cuda:2')
model = ESMC.from_pretrained("esmc_600m", device=device) # or "cpu"

from Bio import SeqIO   
vocab = model.tokenizer.get_vocab()
aa_list = [None]*20
for aa, i in vocab.items():
    if(4 <= i < 24):
        aa_list[i-4] = aa
AA_str = ''.join(aa_list)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv('/home/andrew/GO_interp/data/llps/llps_data.tsv', sep='\t')

In [11]:
from go_ml.masking import mask_range
def gibbs_sampler(seq, mask_func, num_iter=16, batch_size=8):
    seq_ind = model.encode(ESMProtein(sequence=seq)).sequence.to(device)
    mask_batch_sample_l = []
    batch_sample_l = [] 
    bert_eval_l = []
    seq_batch = torch.tile(seq_ind.reshape(1, -1), (batch_size, 1))
    with torch.no_grad():
        for _ in range(num_iter):
            update_batch = mask_func(seq_batch)
            batch_sample_l.append(seq_batch.cpu())
            mask_batch_sample_l.append(update_batch.cpu()) #Save masked batch
            model_eval = model(update_batch)
            bert_eval = model_eval.sequence_logits
            bert_eval = torch.softmax(bert_eval[..., 4:24], dim=2)
            N, L, T = bert_eval.shape
            sample_tokens = torch.multinomial(bert_eval.reshape(N*L, T), num_samples=1).reshape(N, L)+4
            update_batch = update_batch * (update_batch != SEQUENCE_MASK_TOKEN) + sample_tokens * (update_batch == SEQUENCE_MASK_TOKEN)
            bert_eval_l.append(bert_eval.cpu())
            seq_batch = update_batch
    return torch.stack(bert_eval_l), torch.stack(batch_sample_l).cpu(), torch.stack(mask_batch_sample_l).cpu()

In [12]:
from tqdm.notebook import tqdm
def local_gibbs_statistics(seq, segment_len=15, step_size=5, num_iter=16):
    si_l = []
    batch_sample_l = []
    for si in tqdm(range(1, len(seq)+1-segment_len, step_size)):
        ei = si+segment_len
        mask_func = lambda seq_batch: mask_range(seq_batch, si, ei, SEQUENCE_MASK_TOKEN, mut_per=0.15)
        bert_eval, batch_sample, mask_batch_sample = gibbs_sampler(seq, mask_func=mask_func, num_iter=num_iter, batch_size=8)
        si_l.append(si); batch_sample_l.append(batch_sample[..., si:ei])
    return si_l, batch_sample_l

fg_window_stat = local_gibbs_statistics(sequences[0], segment_len=45, step_size=20, num_iter=32)
si_l, gibbs_batch_l = fg_window_stat

  0%|          | 0/25 [00:00<?, ?it/s]

KeyboardInterrupt: 